In [1]:
import json
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings("ignore")


In [10]:
# connect to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
def load_json(json_path):
    with open(json_path, "r") as file:
        raw = file.read()
        data = json.loads(raw.split("```")[-1]) if "```" in raw else json.loads(raw)
    return data

In [29]:
def load_data(csv_path):
    return pd.read_csv(csv_path)

In [30]:
def preprocess_features(df, feature_cfg):
    selected_cols = []
    transformers = []

    for col, cfg in feature_cfg.items():
        if cfg["is_selected"]:
            ftype = cfg["feature_variable_type"]
            method = cfg["feature_details"]["impute_with"]
            value = cfg["feature_details"]["impute_value"]

            imputer = SimpleImputer(strategy="constant", fill_value=value)
            transformers.append((f"imp_{col}", imputer, [col]))
            selected_cols.append(col)

    preprocessor = ColumnTransformer(transformers=transformers)
    X_transformed = preprocessor.fit_transform(df[selected_cols])
    return pd.DataFrame(X_transformed, columns=selected_cols), selected_cols


In [31]:
def generate_features(X, gen_cfg):
    for interaction in gen_cfg.get("explicit_pairwise_interactions", []):
        f1, f2 = interaction.split("/")
        if f1 in X.columns and f2 in X.columns:
            X[f"{f1}_x_{f2}"] = X[f1] * X[f2]
    return X

In [32]:
def reduce_features(method, config, X, y):
    if method == "PCA":
        return PCA(n_components=int(config["num_of_features_to_keep"]))
    elif method == "Tree-based":
        forest = RandomForestRegressor(
            n_estimators=int(config["num_of_trees"]),
            max_depth=int(config["depth_of_trees"]),
            random_state=42,
        )
        return SelectFromModel(forest)
    return "passthrough"

In [33]:
def get_model_and_params(algos_cfg):
    for name, cfg in algos_cfg.items():
        if cfg["is_selected"] and "Regressor" in name:
            if name == "RandomForestRegressor":
                model = RandomForestRegressor()
                params = {
                    "model__n_estimators": list(range(cfg["min_trees"], cfg["max_trees"] + 1, 5)),
                    "model__max_depth": list(range(cfg["min_depth"], cfg["max_depth"] + 1, 5)),
                    "model__min_samples_leaf": list(
                        range(cfg["min_samples_per_leaf_min_value"], cfg["min_samples_per_leaf_max_value"] + 1)
                    ),
                }
                return name, model, params
    return None, None, None


In [34]:
def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)
    print("Evaluation Results:")
    print(f"R2 Score: {r2:.4f}")
    print(f"RMSE: {rmse:.4f}")

In [35]:
# Main Execution
config = load_json("/content/drive/MyDrive/Python Programming/Solution/algoparams_from_ui.json.rtf")
df = load_data("/content/drive/MyDrive/Python Programming/Solution/iris.csv")

cfg = config["design_state_data"]
target_col = cfg["target"]["target"]
prediction_type = cfg["target"]["prediction_type"]
feature_cfg = cfg["feature_handling"]
gen_cfg = cfg["feature_generation"]
red_cfg = cfg["feature_reduction"]
model_cfg = cfg["algorithms"]


ValueError: Failed to parse JSON from RTF. Make sure the file is correctly formatted.

In [ ]:
import re

def load_json(json_path):
    with open(json_path, "r") as file:
        content = file.read()

        # Attempt to extract JSON content from RTF using regex
        try:
            json_text = re.search(r'\\{.*\\}', content, re.DOTALL)
            if json_text:
                json_str = json_text.group().replace('\\', '')
            else:
                json_str = content  # fallback to raw if no RTF formatting
            return json.loads(json_str)
        except Exception as e:
            raise ValueError("Failed to parse JSON from RTF. Make sure the file is correctly formatted.") from e


In [ ]:
X, selected_features = preprocess_features(df, feature_cfg)
y = df[species]
X = generate_features(X, gen_cfg)

In [ ]:
reduction_method = red_cfg.get("feature_reduction_method", "None")
reducer = reduce_features(reduction_method, red_cfg, X, y)

In [ ]:
model_name, model, param_grid = get_model_and_params(model_cfg, prediction_type)
if not model:
    print("No suitable classification model selected in JSON.")
else:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    pipeline = Pipeline(steps=[
        ("scaler", RobustScaler()),
        ("reducer", reducer),
        ("model", model),
    ])
    grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
    grid.fit(X_train, y_train)
    print(f"Model used: {model_name}")
    print(f"Best Parameters: {grid.best_params_}")
    evaluate_model(grid, X_test, y_test)
